# Recast

This notebook recasts the EmoBank likert scores into pairwise comparisons. 

In [1]:
import pandas as pd
import numpy as np
import pairadigm as pdm
from itertools import combinations
import itertools

/Users/mlchrzan/opt/anaconda3/envs/emopair/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("Data/individual_reader_ratings.csv")
df

,id,V,A,D
0,Acephalous-Cant-believe_4_47,3,3,3
1,Acephalous-Cant-believe_4_47,3,3,3
2,Acephalous-Cant-believe_4_47,3,5,3
3,Acephalous-Cant-believe_4_47,3,3,4
4,Acephalous-Cant-believe_4_47,3,3,3
...,...,...,...,...
53050,SemEval_1499,3,3,3
53051,SemEval_1499,3,3,3
53052,SemEval_1499,3,3,3
53053,SemEval_1499,2,4,2


In [3]:
items = pd.unique(df['id']).tolist()

# Generate combinations of length 2
# distinct_pairs is an iterator; wrap in list() to see all at once
pairs = list(combinations(items, 2))

print(f"Total pairs: {len(pairs)}")

Total pairs: 55624878


In [5]:
def get_scores(df, item1_id, item2_id, dimension):
    """
    Returns the scores for two items on a given dimension in df.
    """
    item1_scores = df[df['id'] == item1_id][dimension].tolist()
    item2_scores = df[df['id'] == item2_id][dimension].tolist()
    return item1_scores, item2_scores

# Example
get_scores(df, pairs[0][0], pairs[0][1], 'V')

([3, 3, 3, 3, 3], [2, 3, 3, 3, 3])

In [6]:
def get_pairwise_probability(scores_a, scores_b):
    """
    Calculates P(A > B) using an all-pairs comparison.
    Handles ties by splitting the point (0.5).
    Args:
        scores_a (list or array-like): Scores for item A.
        scores_b (list or array-like): Scores for item B.
    Returns:
        float: Probability that item A is rated higher than item B.
    """
    # Convert to numpy arrays for speed
    a = np.array(scores_a)
    b = np.array(scores_b)
    
    # Create a grid of comparisons using broadcasting
    # This creates a matrix where matrix[i, j] is a[i] - b[j]
    diff_matrix = a[:, np.newaxis] - b
    
    # Count wins (A > B), losses (A < B), and ties (A == B)
    wins = np.sum(diff_matrix > 0)
    ties = np.sum(diff_matrix == 0)
    total_comparisons = a.size * b.size
    
    # Calculate probability: Wins + half of ties
    prob_a_better = (wins + (0.5 * ties)) / total_comparisons
    
    return prob_a_better

# Example Usage
item_a_scores = [4, 5, 4, 5, 3] # Generally high
item_b_scores = [2, 3, 2, 4, 1] # Generally low

prob = get_pairwise_probability(item_a_scores, item_b_scores)

print(f"Probability that Item A is rated higher than Item B: {prob:.2f}")
# Result > 0.5 implies A > B

Probability that Item A is rated higher than Item B: 0.90


In [ ]:
def estimate_batch_size(example_row, sample_n=1000, target_fraction=0.25):
    """
    Estimate a safe batch_size given an example row dict.
    target_fraction: fraction of free RAM to use (0.2-0.5 typical).
    Args:
        example_row (dict): Example row to estimate memory usage.
        sample_n (int): Number of rows to sample for estimation.
        target_fraction (float): Fraction of available memory to use.
    Returns:
        int: Estimated batch size.
    """
    try:
        import psutil
    except Exception:
        raise RuntimeError("psutil required: pip3 install psutil")
    import pandas as pd

    sample = pd.DataFrame([example_row] * sample_n)
    bytes_per_row = sample.memory_usage(deep=True).sum() / sample_n
    avail = psutil.virtual_memory().available
    batch_size = max(1, int((avail * target_fraction) // bytes_per_row))
    return batch_size

# Example usage
example_row = {
    'Dimension': 'V',
    'Item_A': items[0],
    'Item_B': items[1],
    'Probability_A_greater_B': 0.5
}

estimated_batch_size = estimate_batch_size(example_row, sample_n=1000, target_fraction=0.25) 

print("Estimated batch_size:", estimated_batch_size)

Estimated batch_size: 4539415


In [ ]:
# Streamed pairwise computation: use combinations() iterator and write batches to disk
from itertools import combinations
import os

dims = ['V', 'A', 'D']
id_scores = {item: grp[dims].to_dict('list') for item, grp in df.groupby('id')}

outfile = "Data/pairwise_probs.csv"
if os.path.exists(outfile):
    os.remove(outfile)

batch = []
batch_size = estimated_batch_size 
write_header = True

for a, b in combinations(items, 2):
    for dim in dims:
        prob = get_pairwise_probability(id_scores[a][dim], id_scores[b][dim])
        batch.append({
            'Dimension': dim,
            'Item_A': a,
            'Item_B': b,
            'Probability_A_greater_B': prob
        })
    if len(batch) >= batch_size:
        pd.DataFrame(batch).to_csv(outfile, mode='a', header=write_header, index=False)
        write_header = False
        batch.clear()

# write any remaining rows
if batch:
    pd.DataFrame(batch).to_csv(outfile, mode='a', header=write_header, index=False)

In [ ]:
import polars as pl
df_probs = pl.read_csv("Data/pairwise_probs.csv")
df_probs

In [12]:
# Make a new column with a 1/0 indicator if Probability_A_greater_B > 0.5
df_probs = df_probs.with_columns(
    (pl.col("Probability_A_greater_B") > 0.5).cast(pl.Int8).alias("A_better_indicator")
)
df_probs

Dimension,Item_A,Item_B,Probability_A_greater_B,A_better_indicator
str,str,str,f64,i8
"""V""","""Acephalous-Cant-believe_4_47""","""Acephalous-Cant-believe_83_354""",0.6,1
"""A""","""Acephalous-Cant-believe_4_47""","""Acephalous-Cant-believe_83_354""",0.52,1
"""D""","""Acephalous-Cant-believe_4_47""","""Acephalous-Cant-believe_83_354""",0.5,0
"""V""","""Acephalous-Cant-believe_4_47""","""Acephalous-Cant-believe_355_49…",0.3,0
"""A""","""Acephalous-Cant-believe_4_47""","""Acephalous-Cant-believe_355_49…",0.6,1
…,…,…,…,…
"""A""","""SemEval_1497""","""SemEval_1499""",0.5,0
"""D""","""SemEval_1497""","""SemEval_1499""",0.6,1
"""V""","""SemEval_1498""","""SemEval_1499""",0.6,1


In [ ]:
# Generate 3 separate files for V, A, D
for dim in dims:
    dim_df = df_probs.filter(pl.col("Dimension") == dim)
    dim_df.write_csv(f"Data/pairwise_probs_{dim}.csv")